# 이슈 비서 키우기 핸즈온: 코드가 어떻게 자라는가

건설 현장 이슈 비서 하나를 6단계로 키우며 11개 후보(Framework 7, Harness 4)의 코드를 비교한다.
각 단계 파일은 이전 단계 파일을 복사해 기능만 더했으므로, **단계 사이의 diff가 곧 그 기능을 더하는 비용**이다.

| 단계 | 더하는 기능 | 과제 |
| --- | --- | --- |
| L1 | 기본 호출 | 현장 보고 한 건의 공종·심각도 판단 |
| L2 | 도구 | 3층 전기 미해결 이슈 조사, 담당자 추천 |
| L3 | 스킬 | SKILL.md의 사내 규칙대로 주간 리포트 작성 (규칙은 프롬프트에 없다) |
| L4 | 서브에이전트 | L3 + 공정 지연 현황. 공정 도구는 서브에이전트(공정 분석가)만 갖는다 |
| L5 | 메모리 | 세션 1에서 선호를 기억시키고, 새 대화인 세션 2에서 리포트 요청 |
| L6 | 멀티에이전트 구조 | 분석가 → 작성자 → 검토자, 반려되면 작성자가 수정 (L4 위에 얹음) |

**여는 법**: 프로젝트 루트에서 `uv run --with jupyter jupyter lab handson.ipynb`, 또는 VS Code에서 열고 Python 커널을 고른다.
1~3장은 코드와 기존 결과만 읽으므로 API 키 없이 돌아간다. 4장(직접 실행)만 프록시와 의존성이 필요하다.
결과 문서: [docs/05-ladder-results.md](docs/05-ladder-results.md)

## 0. 준비

In [1]:
import difflib
import html
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import HTML, Code, Markdown, display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "bench"))
from run import CANDIDATES, LADDER, LADDER_BASE, RUNS, changed_lines, loc  # noqa: E402  실험 러너의 정의를 그대로 쓴다

STEP_NAMES = {"l1": "기본 호출", "l2": "도구", "l3": "스킬", "l4": "서브에이전트", "l5": "메모리", "l6": "멀티에이전트 구조"}
print("후보:", ", ".join(CANDIDATES))


def src(cand, step):
    return ROOT / CANDIDATES[cand]["src"].format(task=step)


def show(cand, step="l1"):
    """단계 파일 전체를 보여 준다."""
    path = src(cand, step)
    display(Markdown(f"**{path.relative_to(ROOT)}** · {loc(path)}줄 (주석·빈 줄 제외)"))
    display(Code(filename=str(path)))


def diff_html(cand, step):
    """이전 단계 대비 diff. 초록은 더한 줄, 빨강 취소선은 지운 줄."""
    base = LADDER_BASE[step]
    old, new = src(cand, base), src(cand, step)
    rows = []
    for line in difflib.ndiff(old.read_text().splitlines(), new.read_text().splitlines()):
        tag = line[:2]
        if tag == "? ":
            continue
        style = {"+ ": "background:rgba(46,160,67,.28)",
                 "- ": "background:rgba(248,81,73,.22);text-decoration:line-through;opacity:.75"}.get(tag, "")
        rows.append(f'<span style="display:block;{style}">{html.escape(line) or " "}</span>')
    head = (f"<b>{cand}</b> · {new.relative_to(ROOT)} · {base.upper()} 대비 "
            f"<b>+{changed_lines(old, new)}줄</b> (새로 쓰거나 바꾼 줄, 주석·빈 줄 제외)")
    return (f'<div style="flex:1;min-width:0"><div style="margin-bottom:4px">{head}</div>'
            f'<pre style="font-size:12px;line-height:1.45;overflow-x:auto;padding:8px;border:1px solid rgba(128,128,128,.3)">'
            f'{"".join(rows)}</pre></div>')


def diff(cand, step):
    display(HTML(diff_html(cand, step)))


def compare(step, *cands):
    """같은 단계를 후보끼리 나란히 비교한다."""
    display(Markdown(f"### {step.upper()} {STEP_NAMES[step]}"))
    display(HTML('<div style="display:flex;gap:16px">' + "".join(diff_html(c, step) for c in cands) + "</div>"))


def walk(cand):
    """한 후보를 L1부터 L6까지 따라간다."""
    show(cand, "l1")
    for step in LADDER[1:]:
        display(Markdown(f"### {step.upper()} {STEP_NAMES[step]}"))
        diff(cand, step)

후보: langgraph, crewai, strands, adk, openai-agents, pydantic-ai, mastra, claude-agent-sdk, deepagents, pi, flue


## 1. 후보 하나를 골라 L1 → L6 따라가기

`CAND`를 바꿔 다시 실행해 보자. 후보 이름은 위에 출력된 목록에 있다.

In [2]:
CAND = "langgraph"
show(CAND, "l1")

**frameworks/langgraph/l1.py** · 12줄 (주석·빈 줄 제외)

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parents[2] / "shared"))

from ladder import API_KEY, BASE_URL, INSTRUCTIONS, MODEL, prompt, save  # noqa: E402
from langchain.agents import create_agent  # noqa: E402
from langchain_openai import ChatOpenAI  # noqa: E402

agent = create_agent(
    ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY),
    system_prompt=INSTRUCTIONS,
)
result = agent.invoke({"messages": [{"role": "user", "content": prompt("l1")}]})
save(result["messages"][-1].text)

**L2 도구**: Python 프레임워크는 함수를 그대로 넘기거나 데코레이터로 감싼다. 타입 힌트와 docstring이 도구 스키마가 된다.
Claude Agent SDK, Pi, Flue, Mastra는 스키마를 직접 선언해야 해서 줄이 더 는다.

In [3]:
diff(CAND, "l2")

**L3 스킬**: 기본 기능이 있으면 스킬 폴더 경로 하나를 넘긴다.
없으면(LangGraph, Pydantic AI, OpenAI Agents SDK) 스킬 목록을 지시문에 넣고 본문을 `load_skill` 도구로 불러오게 직접 만든다.

In [4]:
diff(CAND, "l3")

**L4 서브에이전트**: 대부분 "하위 에이전트를 도구로 노출"하는 같은 모양이다(`as_tool()`, `AgentTool`, `agents`).
공정 도구는 서브에이전트에게만 준다.

In [5]:
diff(CAND, "l4")

**L5 메모리** (L4 대비): 프로세스 두 개(`remember`, `report`)로 나눠 실행한다. 두 번째 세션의 프롬프트에는 선호가 없어서 저장된 기억을 거쳐야만 통과한다.
파일(CLAUDE.md, AGENTS.md, JSON), 키-값 저장소(SqliteStore 등), LLM 분석+임베딩(CrewAI) 세 갈래가 있다.

In [6]:
diff(CAND, "l5")

**L6 멀티에이전트 구조** (L4 대비): 분석가 → 작성자 → 검토자, 반려 시 최대 2번 수정.
프레임워크는 그래프·워크플로·루프 코드로 순서를 고정하고, 하네스 셋(Claude Agent SDK, Deep Agents, Flue)은 서브에이전트를 정의하고 순서는 지시문으로 맡긴다.

In [7]:
diff(CAND, "l6")

한 번에 보고 싶으면 `walk(CAND)`를 실행한다.

In [8]:
# walk("crewai")

## 2. 같은 단계를 후보끼리 비교하기

발표에서 다룬 대비 네 가지다. 후보를 바꿔 넣어 보자.

**스킬: 기본 지원(CrewAI, +3줄) vs 직접 구현(LangGraph, +15줄)**

In [9]:
compare("l3", "crewai", "langgraph")

### L3 스킬

**서브에이전트: 기본 지원(Deep Agents) vs 기능이 없어 세션을 띄우는 도구를 직접 만든 Pi(+24줄)**

In [10]:
compare("l4", "deepagents", "pi")

### L4 서브에이전트

**메모리: 키-값 저장소(ADK) vs 지침 파일(Claude Agent SDK의 CLAUDE.md)**

In [11]:
compare("l5", "adk", "claude-agent-sdk")

### L5 메모리

**멀티에이전트: 코드로 고정(LangGraph StateGraph) vs 지시문으로 위임(Claude Agent SDK)**

In [12]:
compare("l6", "langgraph", "claude-agent-sdk")

### L6 멀티에이전트 구조

## 3. 숫자로 보기

본 실행(batch `final`, 단계마다 1회)의 집계다. 다시 만들려면 `python3 bench/run.py ladder --batch final`.

In [13]:
display(Markdown((ROOT / "results/summary-ladder-final.md").read_text()))

# 이슈 비서 키우기 (batch: final, 모델: anthropic/claude-opus-5.5)

## 코드 줄 수 (괄호: 이전 단계 대비 새로 쓰거나 바꾼 줄. L5·L6는 L4 대비)

| 후보 | L1 | L2 | L3 | L4 | L5 | L6 |
| --- | --- | --- | --- | --- | --- | --- |
| langgraph | 9 | 11 (+4) | 22 (+15) | 28 (+11) | 40 (+18) | 49 (+32) |
| crewai | 8 | 11 (+5) | 11 (+3) | 14 (+9) | 17 (+7) | 21 (+15) |
| strands | 9 | 10 (+4) | 12 (+5) | 15 (+8) | 28 (+17) | 26 (+18) |
| adk | 19 | 21 (+3) | 24 (+6) | 28 (+9) | 36 (+14) | 33 (+19) |
| openai-agents | 10 | 12 (+5) | 23 (+15) | 25 (+8) | 39 (+17) | 34 (+16) |
| pydantic-ai | 9 | 11 (+4) | 22 (+15) | 27 (+11) | 40 (+16) | 31 (+14) |
| mastra | 7 | 10 (+4) | 12 (+4) | 17 (+9) | 26 (+12) | 34 (+28) |
| claude-agent-sdk | 19 | 33 (+17) | 35 (+5) | 40 (+10) | 42 (+6) | 45 (+14) |
| deepagents | 9 | 10 (+3) | 12 (+4) | 14 (+6) | 18 (+9) | 17 (+8) |
| pi | 17 | 32 (+17) | 32 (+3) | 48 (+24) | 49 (+5) | 51 (+18) |
| flue | 21 | 39 (+19) | 41 (+3) | 50 (+12) | 52 (+4) | 60 (+23) |

## 동작 확인 (단계별 1회)

| 후보 | L1 | L2 | L3 | L4 | L5 | L6 |
| --- | --- | --- | --- | --- | --- | --- |
| langgraph | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| crewai | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| strands | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| adk | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| openai-agents | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| pydantic-ai | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| mastra | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| claude-agent-sdk | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| deepagents | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| pi | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| flue | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |

## 비용($) / LLM 호출 수

| 후보 | L1 | L2 | L3 | L4 | L5 | L6 |
| --- | --- | --- | --- | --- | --- | --- |
| langgraph | 0.002 / 1 | 0.033 / 3 | 0.043 / 3 | 0.077 / 5 | 0.098 / 7 | 0.157 / 11 |
| crewai | 0.002 / 1 | 0.035 / 3 | 0.048 / 3 | 0.087 / 5 | 0.435 / 45 | 0.108 / 7 |
| strands | 0.002 / 1 | 0.034 / 3 | 0.041 / 3 | 0.086 / 5 | 0.100 / 7 | 0.093 / 8 |
| adk | 0.002 / 1 | 0.035 / 3 | 0.074 / 5 | 0.111 / 7 | 0.143 / 8 | 0.225 / 9 |
| openai-agents | 0.002 / 1 | 0.036 / 3 | 0.045 / 3 | 0.087 / 5 | 0.100 / 7 | 0.094 / 7 |
| pydantic-ai | 0.002 / 1 | 0.033 / 3 | 0.044 / 3 | 0.083 / 5 | 0.101 / 7 | 0.145 / 11 |
| mastra | 0.002 / 1 | 0.033 / 3 | 0.050 / 3 | 0.093 / 5 | 0.191 / 8 | 0.191 / 11 |
| claude-agent-sdk | 0.035 / 1 | 0.058 / 4 | 0.062 / 4 | 0.129 / 7 | 0.214 / 9 | 0.409 / 10 |
| deepagents | 0.021 / 1 | 0.045 / 3 | 0.034 / 3 | 0.114 / 5 | 0.167 / 10 | 0.276 / 12 |
| pi | 0.008 / 1 | 0.032 / 3 | 0.036 / 3 | 0.078 / 5 | 0.094 / 8 | 0.088 / 8 |
| flue | 0.002 / 1 | 0.022 / 3 | 0.034 / 3 | 0.061 / 5 | 0.087 / 9 | 0.204 / 11 |


공통 자산(프롬프트, 스킬)도 열어 볼 수 있다. 스킬 규칙을 바꾸면 L3~L6 채점 기준(`tasks/ladder_check.py`)도 같이 바꿔야 한다.

In [14]:
display(Code(filename="shared/skills/site-weekly-report/SKILL.md", language="markdown"))

---
name: site-weekly-report
description: 현장 주간 리포트를 작성할 때 따르는 사내 형식과 규칙. 주간 리포트나 현장 리포트를 써 달라는 요청에 사용한다.
---

# 현장 주간 리포트 작성 규칙

리포트는 마크다운 기호 없이 아래 순서의 일반 텍스트 줄로 쓴다.

1. 첫 줄: `[주간 리포트] <층> <공종 한국어명>` (예: `[주간 리포트] 3F 전기`)
2. `미해결: <건수>건` — 상태가 open 또는 in_progress인 이슈 수
3. `최우선: <이슈 ID>` — 심각도가 가장 높은 미해결 이슈. 같으면 생성일(created_at)이 가장 이른 이슈
4. `담당 추천: <이름>` — 해당 공종 작업자 중 현재 배정 건수(open_assignments)가 가장 적은 사람
5. 공정 지연 정보가 있을 때만: `지연 공종: <공종 한국어명> <지연 폭>%p, ...` — 지연 폭(계획 - 실적)이 10%p 이상인 공종만, 큰 순서로
6. 요약 2~3문장
7. 마지막 줄: `문서번호: WR-<층>-<공종 코드>`

공종 한국어명: electrical=전기, plumbing=설비, structural=골조, finishing=마감, fire_protection=소방
공종 코드: electrical=EL, plumbing=PL, structural=ST, finishing=FN, fire_protection=FP

## 4. 직접 실행해 보기 (선택)

준비물:
1. **프록시**: 모든 LLM 호출이 지나가며 토큰과 비용을 기록한다. 터미널에서 OpenRouter 키를 넣어 띄운다.
   `kg run --secret OPENROUTER_API_KEY -- uv run --python 3.12 bench/proxy.py --max-cost 5`
   (keygate가 없으면 `OPENROUTER_API_KEY`를 환경변수로 준 셸에서 `uv run --python 3.12 bench/proxy.py --max-cost 5`)
2. **의존성**: 실행할 후보만 준비하면 된다. Python 후보는 `uv sync --python 3.12 --project frameworks/<후보>`,
   Mastra는 `pnpm --dir frameworks/mastra install`, 하네스는 `docker build -t agentbench-harness harnesses`와 각 하네스 폴더의 의존성 설치.

비용 감각: L1은 약 $0.002, L4는 $0.06~0.13, L6는 $0.09~0.41이다(Opus 5.5 기준).
아래 셀은 실험 러너(`bench/run.py`)로 한 단계를 실행하고 채점한다.

In [ ]:
CAND, STEP = "langgraph", "l2"
p = subprocess.run([sys.executable, "bench/run.py", "run", STEP, CAND, "--batch", "handson"],
                   cwd=ROOT, capture_output=True, text=True)
print(p.stdout or p.stderr)

가장 최근 실행 결과와 채점 내용을 본다. 아직 직접 실행하지 않았다면 본 실행(final) 결과가 나온다.

In [15]:
def latest(cand, step):
    runs = sorted(RUNS.glob(f"{step}-{cand}-*/result.json"), key=lambda p: p.stat().st_mtime)
    if not runs:
        print("실행 기록이 없다")
        return
    res = json.loads(runs[-1].read_text())
    out = runs[-1].parent / "output.json"
    display(Markdown(f"**{res['run_id']}** (batch {res['batch']}) · 성공 {res['success']} · "
                     f"LLM 호출 {res['llm_calls']}번 · ${res['cost']:.3f}"))
    print({k: v for k, v in res.items() if k not in ("run_id", "batch", "api", "http_errors")})
    if out.exists():
        print("\n--- 에이전트 응답 ---\n" + json.loads(out.read_text())["result"])


latest("langgraph", "l3")

**l3-langgraph-1-1028214109** (batch final) · 성공 True · LLM 호출 3번 · $0.043

{'task': 'l3', 'candidate': 'langgraph', 'rep': 1, 'returncode': 0, 'wall_secs': 12.84, 'success': True, 'checks': {'first_line': True, 'unresolved': True, 'top_priority': True, 'assignee': True, 'doc_number_last': True}, 'agents': 1, 'llm_calls': 3, 'input_tokens': 6610, 'output_tokens': 827, 'cache_read': 0, 'cache_write': 0, 'reasoning_tokens': 14, 'cost': 0.04298, 'first_input_tokens': 1161}

--- 에이전트 응답 ---
[주간 리포트] 3F 전기
미해결: 4건
최우선: ISS-014
담당 추천: 이서연
3층 전기 공종 미해결 이슈는 4건입니다. 진행 중 2건(ISS-014, ISS-024), 미착수 2건(ISS-017, ISS-021)입니다. 최우선 이슈 ISS-014는 A구역 케이블 트레이 12m 구간의 접지 본딩 누락 건으로, 심각도 critical이며 박지훈이 처리하고 있습니다. 같은 critical 등급인 ISS-021(B구역 분전반 결선 오류)은 아직 담당자가 없어, 현재 배정이 1건으로 가장 적은 이서연에게 배정할 것을 추천합니다.
문서번호: WR-3F-EL

참고:
- 심각도가 critical인 미해결 이슈는 ISS-014와 ISS-021 두 건입니다. 이 중 생성일이 더 이른 ISS-014(2026-10-14)를 최우선으로 정했습니다. ISS-021의 생성일은 2026-10-20입니다.
- 공정 지연 정보는 조회할 방법이 없어서 규칙에 따라 지연 공종 줄을 넣지 않았습니다.
- 해결된 이슈 2건(ISS-009, ISS-012)은 미해결 건수에 포함하지 않았습니다.


## 5. 실습 과제

1. **스킬 규칙 바꾸기**: `shared/skills/site-weekly-report/SKILL.md`의 문서번호 형식을 바꾸고, 같은 규칙으로 `tasks/ladder_check.py`의 기대값을 고친 뒤 L3를 다시 실행해 본다. 스킬을 기본 지원하는 후보와 직접 구현한 후보 모두 새 규칙을 따르는지 확인한다.
2. **스킬 직접 구현을 기본 기능으로 바꿔 보기**: LangGraph L3의 `load_skill` 우회 구현과 Deep Agents L3(`skills=`)를 비교하고, 두 방식이 모델에게 보내는 시스템 프롬프트 차이를 `results/proxy/first-requests/<run_id>.json`에서 찾아본다.
3. **L6 통제 방식 바꾸기**: Claude Agent SDK L6는 순서를 지시문에 맡긴다. Pi L6처럼 역할마다 `query()`를 코드로 이어 붙이도록 고쳐 보고, 비용과 호출 수가 어떻게 달라지는지 4장의 셀로 확인한다.